In [2]:
import pandas as pd
import json

# 1. Carregar o JSON
with open('MC1_final_00.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

rounds_data = data['rounds']

# Listas para armazenar as linhas
tabela_rounds = []
tabela_messages = []
tabela_participants = []

# Iterar sobre cada round
for round_id, r in enumerate(rounds_data):
    # --- TABELA 1: ROUNDS (Contexto) ---
    context = r.get('environment_context', {})
    # Prevenção extra caso o context seja null
    if not isinstance(context, dict):
        context = {}
        
    market = context.get('market_snapshot', {})
    if not isinstance(market, dict):
        market = {}
    
    round_row = {
        'round_id': round_id,
        'hour': r.get('hour'),
        'event_narrative': context.get('event_narrative'),
        'event_headline': context.get('event_headline'),
        'stock_price': market.get('stock_price'),
        'percent_change': market.get('percent_change'),
        'market_sentiment': market.get('sentiment'),
        'social_state': context.get('social_state')
    }
    tabela_rounds.append(round_row)
    
    # --- TABELA 2: COMMUNICATIONS (Mensagens) ---
    for msg in r.get('communications', []):
        msg['round_id'] = round_id
        
        # CORREÇÃO AQUI: Pegamos o estado interno de forma segura
        internal_state = msg.get('internal_state')
        
        # Verificamos se ele não é None e se é de fato um dicionário
        if isinstance(internal_state, dict):
            msg['state_reacting'] = internal_state.get('reacting')
            msg['state_rationalizing'] = internal_state.get('rationalizing')
            msg['state_deliberating'] = internal_state.get('deliberating')
        else:
            # Se for nulo, preenchemos com nulo nas colunas novas também
            msg['state_reacting'] = None
            msg['state_rationalizing'] = None
            msg['state_deliberating'] = None
            
        # Limpa a coluna original aninhada (se ela existir)
        if 'internal_state' in msg:
            del msg['internal_state'] 
            
        tabela_messages.append(msg)
        
    # --- TABELA 3: PARTICIPANTS (Participantes) ---
    for p in r.get('participants', []):
        # Proteção caso agent_round_metadata venha como null também
        metadata = p.get('agent_round_metadata')
        if not isinstance(metadata, dict):
            metadata = {}
            
        participant_row = {
            'round_id': round_id,
            'agent_id': p.get('agent_id'),
            'agent_role': p.get('agent_role'),
            'declared_action': p.get('declared_action'),
            'sentiment_at_turn': metadata.get('sentiment_at_turn'),
            'action_classification': metadata.get('action_classification')
        }
        tabela_participants.append(participant_row)

# 2. Converter para DataFrames
df_rounds = pd.DataFrame(tabela_rounds)
df_messages = pd.DataFrame(tabela_messages)
df_participants = pd.DataFrame(tabela_participants)

# 3. Exportar para CSVs
df_rounds.to_csv('./data_rounds.csv', index=False)
df_messages.to_csv('./data_messages.csv', index=False)
df_participants.to_csv('./data_participants.csv', index=False)

print("Processamento concluído! Arquivos gerados com sucesso.")

Processamento concluído! Arquivos gerados com sucesso.
